# Model Training and Validation

Trains linear and XGBoost models using expanding-window cross-validation. Evaluates performance for CORE and WITHRENT variants using AUC, Rank IC, and regression metrics.

In [3]:
# Setup
from pathlib import Path
import json, re, warnings
import numpy as np
import pandas as pd
import xgboost as xgb
import pandas as pd

from xgboost import XGBRegressor, XGBClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, average_precision_score, f1_score, accuracy_score, brier_score_loss
)

try:
    import shap
    HAVE_SHAP = True
except Exception:
    HAVE_SHAP = False
    warnings.warn("[NB5] SHAP not available; skipping SHAP exports.")

# Paths
INTERIM = Path("../data/interim")
REPORTS = Path("../../reports/property")
MODELS  = REPORTS / "models"; MODELS.mkdir(parents=True, exist_ok=True)
TABLES  = REPORTS / "tables"; TABLES.mkdir(parents=True, exist_ok=True)

RUN_TAG = "PROP_v1"   # keep consistent with NB3
np.set_printoptions(suppress=True)
pd.options.display.width = 140

## Load Trainable Datasets

In [6]:
# Load trainables + declare context features

paths = {
    "WITHRENT": INTERIM / f"la_month_trainable_{RUN_TAG}__WITHRENT.parquet",
    "CORE":     INTERIM / f"la_month_trainable_{RUN_TAG}__CORE.parquet",
}

# Load trainable datasets
trainables = {}
for variant, p in paths.items():
    if p.exists():
        df = pd.read_parquet(p)
        print(f"[LOAD] {variant}: {p.name} | rows={len(df):,} | span {df['month'].min().date()} → {df['month'].max().date()}")
        trainables[variant] = df
    else:
        print(f"[SKIP] {variant}: {p.name} not found (run NB3 Cell 9/10).")

assert trainables, "No trainable files found. Re-run NB3 Cell 9/10."

# Context feature lists
# Static LA-level (z-scored)
CTX_STATIC = [
    "fastfood_per_1k_hh_z", "supermkt_per_1k_hh_z",
    "stops_per_km2_z", "stops_per_10k_hh_z",
    "imd_score_mean_z", "airbnb_per_10k_hh_z",
]

# Time-varying context (nationwide IPHRP )
# Use the *non-lagged* names 
CTX_TIME_BASE = [
    "iphrp_yoy_z",   
]

# Columns to lag by 1 month (per LA) to avoid leakage
# Include rent dynamics too (only WITHRENT).
TIME_TO_LAG = [
    "iphrp_yoy_z",
    "rent_yoy", "hpi_yoy", "rent_minus_price_yoy",
]

def add_context_lags(df: pd.DataFrame) -> pd.DataFrame:
    """Create 1M lag (per LA) for selected time features."""
    out = df.sort_values(["LA_code","month"]).copy()
    for col in TIME_TO_LAG:
        if col in out.columns:
            out[f"{col}_l1"] = out.groupby("LA_code", group_keys=False)[col].shift(1)
    return out

# Lagging once per variant 
for v in list(trainables.keys()):
    before = set(trainables[v].columns)
    trainables[v] = add_context_lags(trainables[v])
    after  = set(trainables[v].columns)
    created = sorted(c for c in (after - before) if c.endswith("_l1"))
    if created:
        print(f"[LAGS] {v}: created {len(created)} lag cols → {created}")
    else:
        print(f"[LAGS] {v}: no new lag columns created (check TIME_TO_LAG presence).")


[LOAD] WITHRENT: la_month_trainable_PROP_v1__WITHRENT.parquet | rows=32,436 | span 2016-02-29 → 2025-06-30
[LOAD] CORE: la_month_trainable_PROP_v1__CORE.parquet | rows=113,112 | span 1995-08-31 → 2025-06-30
[LAGS] WITHRENT: created 4 lag cols → ['hpi_yoy_l1', 'iphrp_yoy_z_l1', 'rent_minus_price_yoy_l1', 'rent_yoy_l1']
[LAGS] CORE: created 1 lag cols → ['iphrp_yoy_z_l1']


In [8]:
# verify context columns made it

INTERIM = Path("../data/interim")
ctx_static = INTERIM / "context/context_static.parquet"
ctx_time   = INTERIM / "context/context_time.parquet"

CTX_STATIC_PREFIXES = [
    'fastfood_per_1k_hh','supermkt_per_1k_hh','stops_per_km2','stops_per_10k_hh',
    'imd_score_mean','airbnb_per_10k_hh'
]
CTX_TIME_PREFIXES = ['iphrp_england','iphrp_yoy']

def cols_with_prefix(cols, prefixes):
    cols = list(cols)
    return sorted([c for c in cols for p in prefixes if c.startswith(p)])

# Check context files exist + show sample columns
if ctx_static.exists():
    cs = pd.read_parquet(ctx_static)
    print("[CTX STATIC exists]", ctx_static)
    print("  sample cols:", sorted(cs.columns)[:12])
else:
    print("[CTX STATIC missing]", ctx_static)

if ctx_time.exists():
    ct = pd.read_parquet(ctx_time)
    print("[CTX TIME exists]", ctx_time)
    print("  sample cols:", sorted(ct.columns)[:12])
else:
    print("[CTX TIME missing]", ctx_time)

# Check trainables contain static and lagged time features
for variant, df in trainables.items():
    print(f"\n[{variant}] columns: {len(df.columns)} total")

    static_in_df = cols_with_prefix(df.columns, [f"{p}_z" for p in CTX_STATIC_PREFIXES])
    time_in_df   = cols_with_prefix(df.columns, CTX_TIME_PREFIXES) + \
                   cols_with_prefix(df.columns, [f"{p}_z_l1" for p in CTX_TIME_PREFIXES]) + \
                   [c for c in df.columns if c.endswith("_l1") and c.startswith(("rent_yoy","hpi_yoy","rent_minus_price_yoy"))]

    print("  Static ctx present  :", static_in_df if static_in_df else "none")
    print("  Time ctx present    :", sorted(set(time_in_df)) if time_in_df else "none")

    # Preview of any *_l1 columns
    lag_cols = [c for c in df.columns if c.endswith("_l1")]
    if lag_cols:
        print("  Sample lag cols     :", sorted(lag_cols)[:8])



[CTX STATIC exists] ../data/interim/context/context_static.parquet
  sample cols: ['LA_code', 'LA_name', 'airbnb_per_10k_hh', 'airbnb_per_10k_hh_z', 'area_km2', 'fastfood_per_1k_hh', 'fastfood_per_1k_hh_z', 'households_2021', 'imd_score_mean', 'imd_score_mean_z', 'stops_per_10k_hh', 'stops_per_10k_hh_z']
[CTX TIME exists] ../data/interim/context/context_time.parquet
  sample cols: ['LA_code', 'iphrp_england', 'iphrp_england_z', 'iphrp_yoy', 'iphrp_yoy_z', 'month']

[WITHRENT] columns: 51 total
  Static ctx present  : ['airbnb_per_10k_hh_z', 'fastfood_per_1k_hh_z', 'imd_score_mean_z', 'stops_per_10k_hh_z', 'stops_per_km2_z', 'supermkt_per_1k_hh_z']
  Time ctx present    : ['hpi_yoy_l1', 'iphrp_england', 'iphrp_england_z', 'iphrp_yoy', 'iphrp_yoy_z', 'iphrp_yoy_z_l1', 'rent_minus_price_yoy_l1', 'rent_yoy_l1']
  Sample lag cols     : ['hpi_yoy_l1', 'iphrp_yoy_z_l1', 'rent_minus_price_yoy_l1', 'rent_yoy_l1']

[CORE] columns: 45 total
  Static ctx present  : ['airbnb_per_10k_hh_z', 'fastfoo

## Feature Selection

In [11]:
def pick_features(df: pd.DataFrame, variant: str) -> list[str]:
    """
    Automatically select ALL features except identifiers and targets.

    Includes:
      - All price/return features (ret_*, mom_*, vol_*, etc.)
      - All context features (static + time, raw + z-scored)
      - All rent features (if available)
      - All regime features
    """
    # Exclude identifiers, targets, and metadata
    exclude = {
        "month", "LA_code", "LA_name", "fold", "split", "px_used",
        "y_ret_1m_fwd", "y_ret_3m_fwd", "y_up_1m", "y_up_3m",
        "households_2021",  # Not a feature, just a denominator
    }

    # Get all columns that are not excluded
    features = [c for c in df.columns if c not in exclude]

    # Prefer lagged versions of time features when available
    time_feats_to_check = [
        "iphrp_yoy_z", "iphrp_england_z", "iphrp_yoy", "iphrp_england",
        "rent_yoy", "hpi_yoy", "rent_minus_price_yoy",
    ]

    final_features: list[str] = []

    for feat in features:
        # If this is a non-lagged time feature, check if lagged version exists
        if feat in time_feats_to_check:
            lagged = f"{feat}_l1"
            if lagged in df.columns:
                # Use the lagged version instead of the non-lagged one
                if lagged not in final_features:
                    final_features.append(lagged)
                continue

        # If already a lagged feature, keep it
        if feat.endswith("_l1"):
            final_features.append(feat)
        # If there is no lagged counterpart, keep the original
        elif f"{feat}_l1" not in df.columns:
            final_features.append(feat)
        # else: a lagged version exists; skip this 

    # Remove duplicates and sort
    final_features = sorted(set(final_features))

    assert len(final_features) > 0, f"No features found in dataframe. Columns: {list(df.columns)}"
    return final_features


## Targets, Folds, and Splits

In [14]:
# Targets & fold helpers
REG_TARGETS = ['y_ret_1m_fwd','y_ret_3m_fwd']
CLS_TARGETS = ['y_up_1m','y_up_3m']

def extract_year_from_fold(fold_tag: str) -> int | None:
    m = re.search(r'(\d{4})$', str(fold_tag))
    return int(m.group(1)) if m else None

def split_train_val(df: pd.DataFrame, fold_tag: str):
    """Past-only expanding split: train strictly before val-year."""
    val_year = extract_year_from_fold(fold_tag)
    years = pd.to_datetime(df["month"]).dt.year
    trn = df.loc[years <  val_year].copy()
    val = df.loc[years == val_year].copy()
    return trn, val

def dir_acc(y_true, y_pred) -> float:
    yt = np.asarray(y_true); yp = np.asarray(y_pred)
    return float(np.mean((yt >= 0) == (yp >= 0))) if len(yt) else float("nan")


## Model Parameters and Metrics

In [17]:
# XGB params
def booster_params(task: str) -> dict:
    base = dict(
        tree_method="hist",
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        n_estimators=2000,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
    )
    if task == "reg":
        base.update(objective="reg:squarederror", min_child_weight=5, eval_metric="rmse")
    else:
        base.update(objective="binary:logistic",  min_child_weight=2, eval_metric="auc")
    return base

def early_stop_cb(task: str, rounds: int = 100):
    # maximize AUC for classification; minimize RMSE for regression
    maximize = (task == "cls")
    return [xgb.callback.EarlyStopping(rounds=rounds, save_best=True, maximize=maximize)]

# Metrics
def eval_rank_ic_per_month(y_true, y_pred, months=None):
    """
    Compute Rank IC per month, then average (consistent with stocks pipeline).
    For each month: Spearman correlation between y_true and y_pred within that month.
    If months is None, computes across all samples (fallback).
    """
    from scipy.stats import spearmanr

    if months is None or len(months) != len(y_true):
        # Fallback
        if len(y_true) > 1 and len(np.unique(y_true)) > 1 and len(np.unique(y_pred)) > 1:
            return float(spearmanr(y_true, y_pred)[0])
        return float("nan")

    # Per-month method 
    df = pd.DataFrame({
        'y_true': y_true,
        'y_pred': y_pred,
        'month': pd.to_datetime(months)
    })
    df['month_period'] = df['month'].dt.to_period('M')

    ics = []
    for month_period, group in df.groupby('month_period'):
        if len(group) > 1 and group['y_true'].nunique() > 1 and group['y_pred'].nunique() > 1:
            ic = spearmanr(group['y_true'], group['y_pred'])[0]
            if not np.isnan(ic):
                ics.append(ic)

    return float(np.nanmean(ics)) if ics else float("nan")

def metrics_reg(y_true, y_pred, months=None) -> dict:
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred))) if len(y_true) else float("nan")
    mae  = float(mean_absolute_error(y_true, y_pred)) if len(y_true) else float("nan")
    r2   = float(r2_score(y_true, y_pred)) if len(y_true) else float("nan")
    da   = dir_acc(y_true, y_pred)
    rank_ic = eval_rank_ic_per_month(y_true, y_pred, months)
    return {"rmse": rmse, "mae": mae, "r2": r2, "dir_acc": da, "rank_ic": rank_ic}

def metrics_cls(y_true, y_prob, months=None) -> dict:
    y_true = np.asarray(y_true).astype(int); y_prob = np.asarray(y_prob)
    y_hat = (y_prob >= 0.5).astype(int)
    out = {}
    try:   out["auc"]  = float(roc_auc_score(y_true, y_prob))
    except: out["auc"] = float("nan")
    try:   out["ap"]   = float(average_precision_score(y_true, y_prob))
    except: out["ap"]  = float("nan")
    out["acc"]  = float(accuracy_score(y_true, y_hat))
    out["f1"]   = float(f1_score(y_true, y_hat, zero_division=0))
    out["brier"]= float(brier_score_loss(y_true, y_prob))
    rank_ic = eval_rank_ic_per_month(y_true, y_prob, months)
    out["rank_ic"] = rank_ic
    return out

# Recency weights (exponential decay)
def recency_weights(dates: pd.Series, half_life_months: float = 24.0) -> np.ndarray:
    """Exponential decay weights: more recent samples get higher weight."""
    dates = pd.to_datetime(dates)
    if len(dates) == 0:
        return np.array([])
    max_date = dates.max()
    months_ago = (max_date - dates).dt.days / 30.44  # approximate months
    decay_rate = np.log(2) / half_life_months
    weights = np.exp(-decay_rate * months_ago)
    return weights / weights.sum() * len(weights)  # normalize to sum to n


## Training Loop

In [20]:
# Train models 

EARLY_STOP_ROUNDS = 100

def _to_train_params(params: dict) -> dict:
    """Translate sklearn-style params to core xgb.train params."""
    p = params.copy()
    if "learning_rate" in p:   p["eta"] = p.pop("learning_rate")
    if "reg_lambda"   in p:    p["lambda"] = p.pop("reg_lambda")
    if "n_jobs"       in p:    p["nthread"] = p.pop("n_jobs")
    if "random_state" in p:    p["seed"] = p.pop("random_state")
    return p

def fit_xgb(task: str, params: dict, X_trn, y_trn, X_val, y_val, es_rounds: int = EARLY_STOP_ROUNDS):
    """
    Try sklearn API with ES; if unsupported, try without ES; then fallback to core xgb.train with ES.
    Returns (predict_fn, model_obj, best_iter).
    """
    eval_metric = "auc" if task == "cls" else "rmse"
    params2 = params.copy()
    params2["eval_metric"] = eval_metric

    # sklearn wrapper with ES
    try:
        model = (XGBClassifier if task == "cls" else XGBRegressor)(**params2)
        model.fit(
            X_trn, y_trn,
            eval_set=[(X_val, y_val)],
            early_stopping_rounds=es_rounds,
            verbose=False
        )
        def _pred(X):
            return model.predict_proba(X)[:, 1] if task == "cls" else model.predict(X)
        best_iter = int(getattr(model, "best_iteration", 0))
        return _pred, model, best_iter
    except TypeError:
        pass

    # sklearn wrapper without ES
    try:
        model = (XGBClassifier if task == "cls" else XGBRegressor)(**params2)
        model.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], verbose=False)
        def _pred(X):
            return model.predict_proba(X)[:, 1] if task == "cls" else model.predict(X)
        best_iter = int(getattr(model, "best_iteration", 0))
        return _pred, model, best_iter
    except TypeError:
        pass

    # core API with ES
    dtrn = xgb.DMatrix(X_trn, label=y_trn, feature_names=list(X_trn.columns))
    dval = xgb.DMatrix(X_val, label=y_val, feature_names=list(X_val.columns))
    watch = [(dtrn, "train"), (dval, "eval")]

    train_params = _to_train_params(params2)
    num_boost_round = int(train_params.pop("n_estimators", 2000))

    bst = xgb.train(
        train_params,
        dtrn,
        num_boost_round=num_boost_round,
        evals=watch,
        early_stopping_rounds=es_rounds,
        verbose_eval=False,
    )

    def _pred(X):
        dm = xgb.DMatrix(X, feature_names=list(X.columns))
        return bst.predict(dm)

    best_iter = int(getattr(bst, "best_iteration", getattr(bst, "best_ntree_limit", 0)))
    return _pred, bst, best_iter


# Dry-run summary of which context features are present per variant 
for variant, df in trainables.items():
    features = pick_features(df, variant)

    # Confirmation of context actually included in the feature set
    ctx_used = [c for c in features if any(k in c for k in [
        "fastfood_per_1k_hh", "supermkt_per_1k_hh", "stops_per_km2", "stops_per_10k_hh",
        "imd_score_mean", "airbnb_per_10k_hh"
    ])]
    used_ctx_time = [c for c in features if c.endswith("_l1") and any(k in c for k in ["iphrp", "rent_yoy", "hpi_yoy", "rent_minus_price_yoy"])]

    print(f"\n=== Variant {variant} | {len(features)} features ===")
    print(f"[{variant}] Static context used: {ctx_used or 'none'}")
    print(f"[{variant}] Time context used  : {used_ctx_time or 'none'}")


# Train everything; write models, SHAP, OOF, metrics 
all_metrics = []
all_oof = []

for variant, df in trainables.items():
    features = pick_features(df, variant)
    assert len(features) > 0, "No features found — check pick_features()."

    print(f"\n=== Variant {variant} | {len(features)} features ===")
    folds = sorted([f for f in df['fold'].dropna().unique() if str(f) != "train"])
    print("Folds:", folds)

    #  Regression tasks 
    for target in REG_TARGETS:
        if target not in df.columns:
            print(f"  [SKIP] {target} missing.")
            continue

        task = "reg"
        fold_oof = []
        for fold_tag in folds:
            trn, val = split_train_val(df, fold_tag)
            trn = trn.dropna(subset=[target])
            val = val.dropna(subset=[target])
            if trn.empty or val.empty:
                print(f"  [WARN] Empty split for {fold_tag} — skip.")
                continue

            X_trn = trn[features].astype("float32"); y_trn = trn[target].values
            X_val = val[features].astype("float32"); y_val = val[target].values

                        # - Linear Regression (baseline) -
            # Drop rows with NaNs for linear models
            # Keep original indices to align y and weights
            trn_mask = X_trn.notna().all(axis=1)
            val_mask = X_val.notna().all(axis=1)

            X_trn_clean = X_trn[trn_mask].copy()
            y_trn_clean = y_trn[trn_mask]
            trn_clean = trn[trn_mask].copy()

            X_val_clean = X_val[val_mask].copy()
            y_val_clean = y_val[val_mask]
            val_clean = val[val_mask].copy()

            if len(X_trn_clean) == 0 or len(X_val_clean) == 0:
                print(f"    [SKIP] No valid rows for linear model in {fold_tag} after dropping NaNs (trn={len(X_trn_clean)}, val={len(X_val_clean)})")
                # Still train XGBoost
            else:
                # Scale the clean data
                scaler = StandardScaler()
                X_trn_scaled = pd.DataFrame(
                    scaler.fit_transform(X_trn_clean),
                    index=X_trn_clean.index,
                    columns=X_trn_clean.columns
                )
                X_val_scaled = pd.DataFrame(
                    scaler.transform(X_val_clean),
                    index=X_val_clean.index,
                    columns=X_val_clean.columns
                )

                # Compute recency weights for clean training data
                w_trn_clean = recency_weights(trn_clean["month"], half_life_months=24.0)

                lr = LinearRegression()
                lr.fit(X_trn_scaled, y_trn_clean, sample_weight=w_trn_clean)
                y_pred_lr = lr.predict(X_val_scaled)

                m_lr = metrics_reg(y_val_clean, y_pred_lr, months=val_clean["month"].values)
                m_lr.update(dict(
                    variant=variant, task=task, target=target, fold=fold_tag,
                    n_trn=len(X_trn_clean), n_val=len(X_val_clean), model="logit", best_iter=0
                ))
                all_metrics.append(m_lr)

                fold_oof.append(pd.DataFrame({
                    "variant": variant, "task": task, "target": target, "fold": fold_tag,
                    "model": "logit",
                    "LA_code": val_clean["LA_code"].values, "month": val_clean["month"].values,
                    "y_true": y_val_clean, "y_pred": y_pred_lr
                }))

            # XGBoost
            params = booster_params(task)
            pred_fn, model, best_iter = fit_xgb("reg", params, X_trn, y_trn, X_val, y_val)

            y_pred_xgb = pred_fn(X_val)
            m_xgb = metrics_reg(y_val, y_pred_xgb, months=val["month"].values)
            m_xgb.update(dict(variant=variant, task=task, target=target, fold=fold_tag,
                          n_trn=len(trn), n_val=len(val), model="xgb", best_iter=int(best_iter)))
            all_metrics.append(m_xgb)

            fold_oof.append(pd.DataFrame({
                "variant": variant, "task": task, "target": target, "fold": fold_tag,
                "model": "xgb",
                "LA_code": val["LA_code"].values, "month": val["month"].values,
                "y_true": y_val, "y_pred": y_pred_xgb
            }))

            base = f"{RUN_TAG}_{variant}_{task}_{target}_{fold_tag}"
            model.save_model(MODELS / f"{base}.json")
            (MODELS / f"{base}.features.txt").write_text("\n".join(features))
            meta = {
                "params": getattr(model, "get_xgb_params", lambda: params)(),
                "best_ntree_limit": int(getattr(model, "best_ntree_limit", 0)),
                "best_iteration": int(best_iter),
            }
            (MODELS / f"{base}.meta.json").write_text(json.dumps(meta, indent=2))

            if HAVE_SHAP:
                try:
                    explainer = shap.Explainer(model)
                    sv = explainer(X_val).values  # (n_val, n_feat)
                    shap_abs = np.abs(sv).mean(axis=0)
                    shap_df = (pd.DataFrame({"feature": features, "abs_shap_mean": shap_abs})
                               .sort_values("abs_shap_mean", ascending=False))
                    shap_df.to_parquet(TABLES / f"shap_{base}.parquet", index=False)

                    # Save row-level SHAP for binned plots and recipe tables
                    # Sample up to 200 rows
                    n_sample = min(200, len(X_val))
                    if n_sample > 0:
                        sample_idx = np.random.RandomState(42).choice(len(X_val), size=n_sample, replace=False)
                        shap_rows_df = X_val.iloc[sample_idx].copy().reset_index(drop=True)
                        shap_rows_df["y_true"] = y_val.iloc[sample_idx].values if hasattr(y_val, 'iloc') else y_val[sample_idx]
                        # Add SHAP columns with shap__ prefix
                        for j, feat in enumerate(features):
                            shap_rows_df[f"shap__{feat}"] = sv[sample_idx, j].astype("float32")
                        shap_rows_df.to_parquet(TABLES / f"shap_rows_{base}.parquet", index=False)
                        print(f"    [SHAP] Saved summary + {n_sample} row-level samples → shap_{base}.parquet, shap_rows_{base}.parquet")
                    else:
                        print(f"    [SHAP] Saved summary only (no validation rows) → shap_{base}.parquet")
                except Exception as e:
                    print(f"    [SHAP] Skipped for {fold_tag}: {e}")

        if fold_oof:
            oof_df = pd.concat(fold_oof, ignore_index=True)
            oof_out = TABLES / f"oof_{RUN_TAG}_{variant}_{task}_{target}.parquet"
            oof_df.to_parquet(oof_out, index=False)
            all_oof.append(oof_df)
            print(f" OOF saved → {oof_out} (rows={len(oof_df):,})")

    #  Classification tasks 
    for target in CLS_TARGETS:
        if target not in df.columns:
            print(f"  [SKIP] {target} missing.")
            continue

        task = "cls"
        fold_oof = []
        for fold_tag in folds:
            trn, val = split_train_val(df, fold_tag)
            trn = trn.dropna(subset=[target])
            val = val.dropna(subset=[target])
            if trn.empty or val.empty:
                print(f"  [WARN] Empty split for {fold_tag} — skip.")
                continue

            X_trn = trn[features].astype("float32"); y_trn = trn[target].astype(int).values
            X_val = val[features].astype("float32"); y_val = val[target].astype(int).values

                        # - Logistic Regression (baseline) -
            # Drop rows with NaNs for linear models (they don't handle NaNs)
            # Keep original indices to align y
            trn_mask = X_trn.notna().all(axis=1)
            val_mask = X_val.notna().all(axis=1)

            X_trn_clean = X_trn[trn_mask].copy()
            y_trn_clean = y_trn[trn_mask]
            trn_clean = trn[trn_mask].copy()

            X_val_clean = X_val[val_mask].copy()
            y_val_clean = y_val[val_mask]
            val_clean = val[val_mask].copy()

            if len(X_trn_clean) == 0 or len(X_val_clean) == 0:
                print(f"    [SKIP] No valid rows for linear model in {fold_tag} after dropping NaNs (trn={len(X_trn_clean)}, val={len(X_val_clean)})")
                # Still train XGBoost
            else:
                # Scale the clean data
                scaler = StandardScaler()
                X_trn_scaled = pd.DataFrame(
                    scaler.fit_transform(X_trn_clean),
                    index=X_trn_clean.index,
                    columns=X_trn_clean.columns
                )
                X_val_scaled = pd.DataFrame(
                    scaler.transform(X_val_clean),
                    index=X_val_clean.index,
                    columns=X_val_clean.columns
                )

                # Compute recency weights for clean training data
                w_trn_clean = recency_weights(trn_clean["month"], half_life_months=24.0)

                logit = LogisticRegression(max_iter=2000, C=1.0, solver="lbfgs")
                logit.fit(X_trn_scaled, y_trn_clean, sample_weight=w_trn_clean)
                y_prob_logit = logit.predict_proba(X_val_scaled)[:, 1]

                m_logit = metrics_cls(y_val_clean, y_prob_logit, months=val_clean["month"].values)
                m_logit.update(dict(
                    variant=variant, task=task, target=target, fold=fold_tag,
                    n_trn=len(X_trn_clean), n_val=len(X_val_clean), model="logit", best_iter=0
                ))
                all_metrics.append(m_logit)

                fold_oof.append(pd.DataFrame({
                    "variant": variant, "task": task, "target": target, "fold": fold_tag,
                    "model": "logit",
                    "LA_code": val_clean["LA_code"].values, "month": val_clean["month"].values,
                    "y_true": y_val_clean, "y_prob": y_prob_logit
                }))

            # XGBoost 
            params = booster_params(task)
            # imbalance handling
            pos = float(y_trn.sum()); neg = float(len(y_trn) - y_trn.sum())
            if pos > 0:
                params["scale_pos_weight"] = max(1.0, neg / pos)

            pred_fn, model, best_iter = fit_xgb("cls", params, X_trn, y_trn, X_val, y_val)

            y_prob_xgb = pred_fn(X_val)
            m_xgb = metrics_cls(y_val, y_prob_xgb, months=val["month"].values)
            m_xgb.update(dict(variant=variant, task=task, target=target, fold=fold_tag,
                          n_trn=len(trn), n_val=len(val), model="xgb", best_iter=int(best_iter)))
            all_metrics.append(m_xgb)

            fold_oof.append(pd.DataFrame({
                "variant": variant, "task": task, "target": target, "fold": fold_tag,
                "model": "xgb",
                "LA_code": val["LA_code"].values, "month": val["month"].values,
                "y_true": y_val, "y_prob": y_prob_xgb
            }))

            base = f"{RUN_TAG}_{variant}_{task}_{target}_{fold_tag}"
            model.save_model(MODELS / f"{base}.json")
            (MODELS / f"{base}.features.txt").write_text("\n".join(features))
            meta = {
                "params": getattr(model, "get_xgb_params", lambda: params)(),
                "best_ntree_limit": int(getattr(model, "best_ntree_limit", 0)),
                "best_iteration": int(best_iter),
            }
            (MODELS / f"{base}.meta.json").write_text(json.dumps(meta, indent=2))

            if HAVE_SHAP:
                try:
                    explainer = shap.Explainer(model)
                    sv = explainer(X_val).values
                    shap_abs = np.abs(sv).mean(axis=0)
                    shap_df = (pd.DataFrame({"feature": features, "abs_shap_mean": shap_abs})
                               .sort_values("abs_shap_mean", ascending=False))
                    shap_df.to_parquet(TABLES / f"shap_{base}.parquet", index=False)

                    # Save row-level SHAP for binned plots and recipe tables
                    # Sample up to 200 rows to keep file sizes manageable
                    n_sample = min(200, len(X_val))
                    if n_sample > 0:
                        sample_idx = np.random.RandomState(42).choice(len(X_val), size=n_sample, replace=False)
                        shap_rows_df = X_val.iloc[sample_idx].copy().reset_index(drop=True)
                        shap_rows_df["y_true"] = y_val.iloc[sample_idx].values if hasattr(y_val, 'iloc') else y_val[sample_idx]
                        # Add SHAP columns with shap__ prefix
                        for j, feat in enumerate(features):
                            shap_rows_df[f"shap__{feat}"] = sv[sample_idx, j].astype("float32")
                        shap_rows_df.to_parquet(TABLES / f"shap_rows_{base}.parquet", index=False)
                        print(f"    [SHAP] Saved summary + {n_sample} row-level samples → shap_{base}.parquet, shap_rows_{base}.parquet")
                    else:
                        print(f"    [SHAP] Saved summary only (no validation rows) → shap_{base}.parquet")
                except Exception as e:
                    print(f"    [SHAP] Skipped for {fold_tag}: {e}")

        if fold_oof:
            oof_df = pd.concat(fold_oof, ignore_index=True)
            oof_out = TABLES / f"oof_{RUN_TAG}_{variant}_{task}_{target}.parquet"
            oof_df.to_parquet(oof_out, index=False)
            all_oof.append(oof_df)
            print(f"  ✓ OOF saved → {oof_out} (rows={len(oof_df):,})")

# Aggregate & save metrics 
if all_metrics:
    metrics_df = pd.DataFrame(all_metrics)
    met_out = TABLES / f"metrics_{RUN_TAG}_property.parquet"
    csv_out = TABLES / f"metrics_{RUN_TAG}_property.csv"
    metrics_df.to_parquet(met_out, index=False)
    metrics_df.to_csv(csv_out, index=False)
    print(f"\nMetrics saved → {met_out} and {csv_out}")

if all_oof:
    oof_all = pd.concat(all_oof, ignore_index=True)
    oof_all_out = TABLES / f"oof_{RUN_TAG}_ALL_property.parquet"
    oof_all.to_parquet(oof_all_out, index=False)
    print(f"Combined OOF saved → {oof_all_out} (rows={len(oof_all):,})")




=== Variant WITHRENT | 37 features ===
[WITHRENT] Static context used: ['airbnb_per_10k_hh', 'airbnb_per_10k_hh_z', 'fastfood_per_1k_hh', 'fastfood_per_1k_hh_z', 'imd_score_mean', 'imd_score_mean_z', 'stops_per_10k_hh', 'stops_per_10k_hh_z', 'stops_per_km2', 'stops_per_km2_z', 'supermkt_per_1k_hh', 'supermkt_per_1k_hh_z']
[WITHRENT] Time context used  : ['hpi_yoy_l1', 'iphrp_yoy_z_l1', 'rent_minus_price_yoy_l1', 'rent_yoy_l1']

=== Variant CORE | 34 features ===
[CORE] Static context used: ['airbnb_per_10k_hh', 'airbnb_per_10k_hh_z', 'fastfood_per_1k_hh', 'fastfood_per_1k_hh_z', 'imd_score_mean', 'imd_score_mean_z', 'stops_per_10k_hh', 'stops_per_10k_hh_z', 'stops_per_km2', 'stops_per_km2_z', 'supermkt_per_1k_hh', 'supermkt_per_1k_hh_z']
[CORE] Time context used  : ['iphrp_yoy_z_l1']

=== Variant WITHRENT | 37 features ===
Folds: ['A_val2020', 'B_val2022', 'C_val2023']
    [SHAP] Saved summary + 200 row-level samples → shap_PROP_v1_WITHRENT_reg_y_ret_1m_fwd_A_val2020.parquet, shap_row

## Aggregate Metrics

In [23]:
# Metrics leaderboard

TABLES = Path("../../reports/property/tables")
RUN_TAG = "PROP_v1"

# Load the metrics parquet 
mp = TABLES / f"metrics_{RUN_TAG}.parquet"
if not mp.exists():
    mp = TABLES / f"metrics_{RUN_TAG}_property.parquet"
metrics_df = pd.read_parquet(mp)

# Regression best-of per (variant, target) 
reg_cols = [c for c in ["rmse","mae","dir_acc","r2","fold","n_trn","n_val","best_iter"] if c in metrics_df.columns]
reg = metrics_df[metrics_df["task"]=="reg"].copy()

# sort key
reg = reg.sort_values(
    by=[c for c in ["rmse","mae","dir_acc","r2","fold"] if c in reg.columns],
    ascending=[True, True, False, False, True][:len([c for c in ["rmse","mae","dir_acc","r2","fold"] if c in reg.columns])]
)
leader_reg = reg.drop_duplicates(subset=["variant","task","target"], keep="first")

# Classification best-of per (variant, target) 
cls_cols = [c for c in ["auc","ap","acc","f1","brier","fold","n_trn","n_val","best_iter"] if c in metrics_df.columns]
cls = metrics_df[metrics_df["task"]=="cls"].copy()


# Prioritise auc, then ap, then brier (ascending), then acc, f1
order = [c for c in ["auc","ap","brier","acc","f1","fold"] if c in cls.columns]
asc   = []
for c in order:
    if c == "brier": asc.append(True)   # lower is better
    elif c == "fold": asc.append(True)  # stable
    else:             asc.append(False) # higher is better
cls = cls.sort_values(by=order, ascending=asc)
leader_cls = cls.drop_duplicates(subset=["variant","task","target"], keep="first")

#Print 
print("\nBest fold per (variant, reg):")
display(leader_reg[["rmse","mae","r2","dir_acc","variant","task","target","fold","n_trn","n_val","best_iter"]].round(4))

print("\n Best fold per (variant, cls):")
keep = [c for c in ["acc","f1","brier","auc","ap","variant","task","target","fold","n_trn","n_val","best_iter"] if c in leader_cls.columns]
display(leader_cls[keep].round(4))

print("\n Combined best-of leaderboard: ")
leader = pd.concat([leader_cls, leader_reg], ignore_index=True, sort=False)
cols = [c for c in [
    "variant","task","target","fold",
    "rmse","mae","r2","dir_acc",
    "auc","ap","acc","f1","brier",
    "n_trn","n_val","best_iter"
] if c in leader.columns]
display(leader[cols].round(4))



Best fold per (variant, reg):


,rmse,mae,r2,dir_acc,variant,task,target,fold,n_trn,n_val,best_iter
27,0.0643,0.0472,0.0647,0.5656,CORE,reg,y_ret_1m_fwd,B_val2022,99881,3780,0
5,0.0672,0.0492,0.1400,0.6022,WITHRENT,reg,y_ret_1m_fwd,C_val2023,23825,3444,0
9,0.0681,0.0512,0.1706,0.6130,WITHRENT,reg,y_ret_3m_fwd,B_val2022,20381,3444,0
31,0.0683,0.0517,0.1123,0.6093,CORE,reg,y_ret_3m_fwd,A_val2020,92320,3780,0



 Best fold per (variant, cls):


,acc,f1,brier,auc,ap,variant,task,target,fold,n_trn,n_val,best_iter
21,0.6228,0.5180,0.2913,0.7649,0.8002,WITHRENT,cls,y_up_3m,B_val2022,20381,3444,0
45,0.6751,0.6395,0.2325,0.7540,0.8039,CORE,cls,y_up_3m,B_val2022,99881,3780,0
17,0.5981,0.5492,0.2549,0.6337,0.5644,WITHRENT,cls,y_up_1m,C_val2023,23825,3444,0
36,0.5814,0.5960,0.2435,0.6274,0.6944,CORE,cls,y_up_1m,A_val2020,2068,528,0



 Combined best-of leaderboard: 


,variant,task,target,fold,rmse,mae,r2,dir_acc,auc,ap,acc,f1,brier,n_trn,n_val,best_iter
0,WITHRENT,cls,y_up_3m,B_val2022,NaN,NaN,NaN,NaN,0.7649,0.8002,0.6228,0.5180,0.2913,20381,3444,0
1,CORE,cls,y_up_3m,B_val2022,NaN,NaN,NaN,NaN,0.7540,0.8039,0.6751,0.6395,0.2325,99881,3780,0
2,WITHRENT,cls,y_up_1m,C_val2023,NaN,NaN,NaN,NaN,0.6337,0.5644,0.5981,0.5492,0.2549,23825,3444,0
3,CORE,cls,y_up_1m,A_val2020,NaN,NaN,NaN,NaN,0.6274,0.6944,0.5814,0.5960,0.2435,2068,528,0
4,CORE,reg,y_ret_1m_fwd,B_val2022,0.0643,0.0472,0.0647,0.5656,NaN,NaN,NaN,NaN,NaN,99881,3780,0
5,WITHRENT,reg,y_ret_1m_fwd,C_val2023,0.0672,0.0492,0.1400,0.6022,NaN,NaN,NaN,NaN,NaN,23825,3444,0
6,WITHRENT,reg,y_ret_3m_fwd,B_val2022,0.0681,0.0512,0.1706,0.6130,NaN,NaN,NaN,NaN,NaN,20381,3444,0
7,CORE,reg,y_ret_3m_fwd,A_val2020,0.0683,0.0517,0.1123,0.6093,NaN,NaN,NaN,NaN,NaN,92320,3780,0
